<a href="https://colab.research.google.com/github/rezzz59/Sentimen-Analysis-Aplikasi-Grab/blob/main/grab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%pip install google-play-scrapper

In [3]:
import sys
!{sys.executable} -m pip install google-play-scraper



In [4]:
import pandas as pd
from google_play_scraper import Sort, reviews

result, countinuation_token = reviews(
    'com.grabtaxi.passenger',
    lang = 'id',
    country = 'id',
    sort = Sort.NEWEST,
    count= 51000,
)

df = pd.DataFrame(result)

df = df[['content', 'score']]

df.to_csv('grab_review_raw.csv', index = False)

In [5]:
%pip install sastrawi

In [6]:
import re
import string
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

slang_dict = {
    "ga": "tidak", "gak": "tidak", "gakk": "tidak", "nggak": "tidak",
    "yg": "yang", "dr": "dari", "bgt": "banget", "kl": "kalau",
    "udh": "sudah", "udah": "sudah", "aja": "saja", "jd": "jadi",
    "tp": "tapi", "pake": "pakai", "sdh": "sudah", "aja": "saja",
    "dapet": "dapat", "ilang": "hilang", "lemot": "lambat", "gercep": "cepat",
}

def clean_text(text):
  text = text.lower()

  #menghapus link, mention dan tag
  text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
  text = text = re.sub(r'@\w+|\#', '', text)

  #menghapus angka dan tanda bca
  text = text.translate(str.maketrans('', '', string.punctuation))
  text = re.sub(r'\d+', '', text)

  #menghapus emoji
  text = text.encode('ascii', 'ignore').decode('ascii')

  #normalisasi kata & tokenizing
  words = text.split()
  cleaned_words = [slang_dict.get(w, w) for w in words]

  return " ".join(cleaned_words)

df['content_cleaned'] = df['content'].apply(clean_text)
df = df[df['content_cleaned'] != '']


In [7]:
def labeling(score):
  if score > 3:
    return "positif"
  if score <= 2:
    return 'negatif'
  else:
    return 'netral'

df['label'] = df['score'].apply(labeling)

In [8]:
df['label'].value_counts()

,count
label,
positif,38064
negatif,10532
netral,1612


In [9]:
from sklearn.utils import resample
target_samples = 8977

df_pos_bal = df[df['label'] == 'positif'].sample(target_samples, random_state=42)
df_neg_bal = df[df['label'] == 'negatif']
df_neu_bal = resample(df[df['label'] == 'netral'], replace=True, n_samples=target_samples, random_state=42)

df_final = pd.concat([df_pos_bal, df_neg_bal, df_neu_bal])

In [10]:
df_final

,content,score,content_cleaned,label
45552,mantappp,5,mantappp,positif
33025,sangat membatu dalam segala hal,5,sangat membatu dalam segala hal,positif
41856,Bagus,5,bagus,positif
21528,bersih dan nyaman,5,bersih dan nyaman,positif
25374,cepat .,5,cepat,positif
...,...,...,...,...
28567,"bagus, penjemputan nya cepat",3,bagus penjemputan nya cepat,netral
41118,Udah bagus sih... Cuman masih aja ada lag pas ...,3,sudah bagus sih cuman masih saja ada lag pas a...,netral
23334,i like,3,i like,netral
20674,susah dapet driver klo jam malam padahal gw be...,3,susah dapat driver klo jam malam padahal gw be...,netral


In [11]:
df_final['label'].value_counts()

,count
label,
negatif,10532
positif,8977
netral,8977


In [12]:
df_final[['content_cleaned', 'score', 'label']]

,content_cleaned,score,label
45552,mantappp,5,positif
33025,sangat membatu dalam segala hal,5,positif
41856,bagus,5,positif
21528,bersih dan nyaman,5,positif
25374,cepat,5,positif
...,...,...,...
28567,bagus penjemputan nya cepat,3,netral
41118,sudah bagus sih cuman masih saja ada lag pas a...,3,netral
23334,i like,3,netral
20674,susah dapat driver klo jam malam padahal gw be...,3,netral


In [13]:
from sklearn.model_selection import train_test_split

X = df_final['content_cleaned'].astype(str).values
y = pd.get_dummies(df_final['label']).astype(int).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)


In [14]:
y

array([[0, 0, 1],
       [0, 0, 1],
       [0, 0, 1],
       ...,
       [0, 1, 0],
       [0, 1, 0],
       [0, 1, 0]])

In [15]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=5000, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

#menyamakan panjang kalimat ada yang panjang dan pendek, sehingga disamakan menjadi 100 kata
X_train_pad = pad_sequences(X_train_seq, maxlen=100, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=100, padding='post', truncating='post')

In [16]:
df_final

,content,score,content_cleaned,label
45552,mantappp,5,mantappp,positif
33025,sangat membatu dalam segala hal,5,sangat membatu dalam segala hal,positif
41856,Bagus,5,bagus,positif
21528,bersih dan nyaman,5,bersih dan nyaman,positif
25374,cepat .,5,cepat,positif
...,...,...,...,...
28567,"bagus, penjemputan nya cepat",3,bagus penjemputan nya cepat,netral
41118,Udah bagus sih... Cuman masih aja ada lag pas ...,3,sudah bagus sih cuman masih saja ada lag pas a...,netral
23334,i like,3,i like,netral
20674,susah dapet driver klo jam malam padahal gw be...,3,susah dapat driver klo jam malam padahal gw be...,netral


In [17]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D, Bidirectional

model = Sequential([
    Embedding(input_dim=10000, output_dim=128), # Sesuai num_words
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(64)),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:


# Callback untuk berhenti otomatis jika akurasi sudah memuaskan
class myCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        if(logs.get('val_accuracy') > 0.88):
            print("\nAkurasi sudah mencapai 93%")
            self.model.stop_training = True

callbacks = myCallback()

# Mulai Pelatihan
history = model.fit(
    X_train_pad, y_train,
    epochs=40,
    batch_size=64,
    validation_data=(X_test_pad, y_test),
    callbacks=[callbacks],
    verbose=1
)

Epoch 1/40
357/357 ━━━━━━━━━━━━━━━━━━━━ 96s 268ms/step - accuracy: 0.7422 - loss: 0.6423 - val_accuracy: 0.7987 - val_loss: 0.5419
Epoch 2/40
357/357 ━━━━━━━━━━━━━━━━━━━━ 101s 282ms/step - accuracy: 0.8431 - loss: 0.4559 - val_accuracy: 0.8299 - val_loss: 0.4787
Epoch 3/40
357/357 ━━━━━━━━━━━━━━━━━━━━ 95s 265ms/step - accuracy: 0.8722 - loss: 0.3845 - val_accuracy: 0.8406 - val_loss: 0.4372
Epoch 4/40
357/357 ━━━━━━━━━━━━━━━━━━━━ 94s 264ms/step - accuracy: 0.8885 - loss: 0.3311 - val_accuracy: 0.8564 - val_loss: 0.4169
Epoch 5/40
357/357 ━━━━━━━━━━━━━━━━━━━━ 142s 264ms/step - accuracy: 0.8962 - loss: 0.3049 - val_accuracy: 0.8501 - val_loss: 0.4210
Epoch 6/40
357/357 ━━━━━━━━━━━━━━━━━━━━ 95s 265ms/step - accuracy: 0.9061 - loss: 0.2775 - val_accuracy: 0.8585 - val_loss: 0.4233
Epoch 7/40
357/357 ━━━━━━━━━━━━━━━━━━━━ 95s 265ms/step - accuracy: 0.9127 - loss: 0.2558 - val_accuracy: 0.8617 - val_loss: 0.4285
Epoch 8/40
357/357 ━━━━━━━━━━━━━━━━━━━━ 141s 264ms/step - accuracy: 0.9150 - loss